# 🏪 Cerebal Kaggle — Solari Hub Order Volume Forecasting
### Minimizing RMSLE | LightGBM + XGBoost + CatBoost Ensemble

**Strategy:**
- Predict 0 for all `IsOpen=0` rows (always true in train, 100% accurate)
- Model only `IsOpen=1` rows with lag/rolling features
- Train on `log1p(OrderVolume)` → equivalent to directly minimizing RMSLE
- Ensemble LightGBM + XGBoost + CatBoost with TimeSeriesSplit CV

## 📦 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

pd.set_option('display.max_columns', 50)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
})
PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657', '#79c0ff']
SEED = 42
np.random.seed(SEED)

DATA_PATH = './Dataset/'   # <-- adjust if needed
print('Imports done')

## 📂 2. Load Data

In [ ]:
train  = pd.read_csv(DATA_PATH + 'orders_train.csv')
test   = pd.read_csv(DATA_PATH + 'orders_test.csv')
meta   = pd.read_csv(DATA_PATH + 'hub_metadata.csv')
sample = pd.read_csv(DATA_PATH + 'sample_submission.csv')

train['Date'] = pd.to_datetime(train['Date'])
test['Date']  = pd.to_datetime(test['Date'])

print(f'Train : {train.shape[0]:>7,} rows x {train.shape[1]} cols')
print(f'Test  : {test.shape[0]:>7,} rows x {test.shape[1]} cols')
print(f'Meta  : {meta.shape[0]:>7,} rows x {meta.shape[1]} cols')
print(f'Train period : {train["Date"].min().date()} to {train["Date"].max().date()}')
print(f'Test  period : {test["Date"].min().date()} to {test["Date"].max().date()}')
print(f'Unique hubs  : {train["HubID"].nunique()}')
train.head(3)

In [ ]:
meta.head(3)

## 🔍 3. Exploratory Data Analysis

### 3.1 Missing Values

In [ ]:
def missing_report(df, name):
    miss = df.isnull().sum()
    pct = (miss / len(df) * 100).round(2)
    report = pd.DataFrame({'Missing': miss, 'Pct%': pct})
    report = report[report['Missing'] > 0].sort_values('Missing', ascending=False)
    print(f'--- {name} ---')
    print('  No missing values!' if report.empty else report.to_string())
    print()

missing_report(train, 'orders_train')
missing_report(test,  'orders_test')
missing_report(meta,  'hub_metadata')

### 3.2 Target Distribution

In [ ]:
print('=== OrderVolume Summary ===')
print(train['OrderVolume'].describe())
print(f'Zeros: {(train["OrderVolume"]==0).sum():,} ({(train["OrderVolume"]==0).mean()*100:.1f}%)')

# Critical check
nonzero_closed = train[(train['IsOpen'] == 0) & (train['OrderVolume'] > 0)]
print(f'\nIsOpen=0 with OrderVolume>0: {len(nonzero_closed)} rows')
print('-> All zeros when closed. Predict 0 directly for IsOpen=0 in test.')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].hist(train['OrderVolume'], bins=80, color=PALETTE[0], alpha=0.85, edgecolor='none')
axes[0].set_title('Raw OrderVolume (all rows)')
axes[0].grid(True, alpha=0.4)

open_vol = train[train['IsOpen'] == 1]['OrderVolume']
axes[1].hist(open_vol, bins=80, color=PALETTE[1], alpha=0.85, edgecolor='none')
axes[1].set_title('OrderVolume (IsOpen=1 only)')
axes[1].grid(True, alpha=0.4)

axes[2].hist(np.log1p(open_vol), bins=80, color=PALETTE[3], alpha=0.85, edgecolor='none')
axes[2].set_title('log1p(OrderVolume) — Model Target')
axes[2].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

### 3.3 Seasonality & Business Patterns

In [ ]:
train_open = train[train['IsOpen'] == 1].copy()
train_open['Month'] = train_open['Date'].dt.month

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

# Weekday
wday_means = train_open.groupby('Weekday')['OrderVolume'].mean()
axes[0,0].bar(wday_means.index, wday_means.values, color=PALETTE[0], alpha=0.85)
axes[0,0].set_title('Mean OrderVolume by Weekday')
axes[0,0].set_xlabel('Weekday (1=Mon, 7=Sun)')
axes[0,0].grid(True, axis='y', alpha=0.4)

# Month
month_means = train_open.groupby('Month')['OrderVolume'].mean()
axes[0,1].bar(month_means.index, month_means.values, color=PALETTE[1], alpha=0.85)
axes[0,1].set_title('Mean OrderVolume by Month')
axes[0,1].set_xlabel('Month')
axes[0,1].grid(True, axis='y', alpha=0.4)

# Promo
promo_means = train_open.groupby('PromoActive')['OrderVolume'].mean()
axes[0,2].bar(['No Promo', 'Promo Active'], promo_means.values, color=[PALETTE[2], PALETTE[3]], alpha=0.85)
axes[0,2].set_title('Promo Effect on OrderVolume')
axes[0,2].grid(True, axis='y', alpha=0.4)

# Holiday
hol_means = train_open.groupby('RegionalHoliday')['OrderVolume'].mean()
labels_h = ['None', 'Public', 'Spring/Rel', 'Winter']
axes[1,0].bar(labels_h[:len(hol_means)], hol_means.values, color=PALETTE[:4], alpha=0.85)
axes[1,0].set_title('OrderVolume by Holiday Type')
axes[1,0].grid(True, axis='y', alpha=0.4)

# School closure
school_means = train_open.groupby('SchoolClosureFlag')['OrderVolume'].mean()
axes[1,1].bar(['No Closure', 'School Closed'], school_means.values, color=[PALETTE[4], PALETTE[5]], alpha=0.85)
axes[1,1].set_title('School Closure Effect')
axes[1,1].grid(True, axis='y', alpha=0.4)

# AppSessions scatter
samp = train_open.sample(4000, random_state=SEED)
corr = train_open[['AppSessions','OrderVolume']].corr().iloc[0,1]
axes[1,2].scatter(samp['AppSessions'], samp['OrderVolume'], alpha=0.25, color=PALETTE[0], s=6)
axes[1,2].set_title(f'AppSessions vs OrderVolume (r={corr:.3f})')
axes[1,2].set_xlabel('AppSessions')
axes[1,2].set_ylabel('OrderVolume')
axes[1,2].grid(True, alpha=0.3)

plt.suptitle('EDA: Business Patterns & Drivers', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 3.4 Hub Metadata Analysis

In [ ]:
train_meta = train_open.merge(meta, on='HubID', how='left')

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for ax, col, label in zip(
    axes,
    ['HubFormat', 'AssortmentTier', 'LoyaltyProgram'],
    ['HubFormat', 'AssortmentTier', 'Loyalty Program']
):
    means = train_meta.groupby(col)['OrderVolume'].mean()
    counts = meta[col].value_counts().sort_index()
    bars = ax.bar(means.index.astype(str), means.values, color=PALETTE, alpha=0.85)
    ax.set_title(f'Mean OrderVolume by {label}')
    ax.grid(True, axis='y', alpha=0.4)
    for bar, cnt, val in zip(bars, counts.values, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 50,
                f'n={cnt}', ha='center', fontsize=8, alpha=0.8)

plt.tight_layout()
plt.show()

print('CompetitorDistance stats:')
print(meta['CompetitorDistance'].describe())
print(f'Missing: {meta["CompetitorDistance"].isna().sum()}')

### 3.5 Time Series: Overall Trend

In [ ]:
daily = train_open.groupby('Date')['OrderVolume'].mean().reset_index()
daily['rolling_7']  = daily['OrderVolume'].rolling(7, center=True).mean()
daily['rolling_28'] = daily['OrderVolume'].rolling(28, center=True).mean()

fig, ax = plt.subplots(figsize=(18, 4))
ax.plot(daily['Date'], daily['OrderVolume'], color=PALETTE[0], alpha=0.3, linewidth=0.6, label='Daily mean')
ax.plot(daily['Date'], daily['rolling_7'],  color=PALETTE[1], linewidth=1.5, label='7-day rolling mean')
ax.plot(daily['Date'], daily['rolling_28'], color=PALETTE[3], linewidth=2.0, label='28-day rolling mean')
ax.set_title('Average Daily OrderVolume Across All Open Hubs (2013–2015)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Mean OrderVolume')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
num_cols = ['OrderVolume', 'AppSessions', 'IsOpen', 'PromoActive',
            'RegionalHoliday', 'SchoolClosureFlag', 'Weekday']
corr = train[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 9})
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 🔧 4. Feature Engineering

In [ ]:
def count_loyalty_intervals(s):
    if pd.isna(s): return 0
    return len(s.split(','))


def engineer_features(df, meta):
    df = df.copy().sort_values(['HubID', 'Date']).reset_index(drop=True)

    # Merge metadata
    df = df.merge(meta, on='HubID', how='left')

    # Date features
    df['Year']       = df['Date'].dt.year
    df['Month']      = df['Date'].dt.month
    df['DayOfMonth'] = df['Date'].dt.day
    df['DayOfYear']  = df['Date'].dt.dayofyear
    df['Week']       = df['Date'].dt.isocalendar().week.astype(int)
    df['Quarter']    = df['Date'].dt.quarter
    df['IsWeekend']  = (df['Weekday'] >= 6).astype(int)

    # Cyclical encoding
    df['weekday_sin'] = np.sin(2 * np.pi * df['Weekday'] / 7)
    df['weekday_cos'] = np.cos(2 * np.pi * df['Weekday'] / 7)
    df['month_sin']   = np.sin(2 * np.pi * df['Month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['Month'] / 12)
    df['doy_sin']     = np.sin(2 * np.pi * df['DayOfYear'] / 365)
    df['doy_cos']     = np.cos(2 * np.pi * df['DayOfYear'] / 365)

    # Metadata features
    df['CompetitorDistance_log'] = np.log1p(
        df['CompetitorDistance'].fillna(df['CompetitorDistance'].median()))
    df['HasCompetitor'] = df['CompetitorOpenSinceYear'].notna().astype(int)
    df['LoyaltyIntervalCount'] = df['LoyaltyProgramInterval'].apply(count_loyalty_intervals)

    # Competitor tenure in months
    df['CompetitorAge_months'] = (
        (df['Year'] - df['CompetitorOpenSinceYear'].fillna(df['Year'])) * 12 +
        (df['Month'] - df['CompetitorOpenSinceMonth'].fillna(df['Month']))
    ).clip(lower=0)

    # Loyalty tenure in weeks
    df['LoyaltyAge_weeks'] = (
        (df['Year'] - df['LoyaltyProgramSinceYear'].fillna(df['Year'])) * 52 +
        (df['Week'] - df['LoyaltyProgramSinceWeek'].fillna(df['Week']))
    ).clip(lower=0)

    # Interaction features
    df['AnyHoliday']        = (df['RegionalHoliday'] > 0).astype(int)
    df['Promo_x_Weekend']   = df['PromoActive'] * df['IsWeekend']
    df['Promo_x_Holiday']   = df['PromoActive'] * df['AnyHoliday']
    df['Promo_x_School']    = df['PromoActive'] * df['SchoolClosureFlag']
    df['Holiday_x_Weekend'] = df['AnyHoliday'] * df['IsWeekend']
    df['School_x_Weekend']  = df['SchoolClosureFlag'] * df['IsWeekend']

    return df


print('Engineering train features...')
train_fe = engineer_features(train, meta)
print('Engineering test features...')
test_fe  = engineer_features(test, meta)
print(f'Done. Train: {train_fe.shape}, Test: {test_fe.shape}')

### 4.1 Lag & Rolling Features

In [ ]:
def add_lag_features(train_df, test_df):
    test_tmp = test_df.copy()
    if 'OrderVolume' not in test_tmp.columns:
        test_tmp['OrderVolume'] = np.nan
    if 'AppSessions' not in test_tmp.columns:
        test_tmp['AppSessions'] = np.nan

    full = pd.concat([
        train_df[['HubID','Date','OrderVolume','AppSessions','IsOpen']],
        test_tmp[['HubID','Date','OrderVolume','AppSessions','IsOpen']]
    ], ignore_index=True).sort_values(['HubID','Date']).reset_index(drop=True)

    LAG_DAYS     = [7, 14, 21, 28, 42, 56]
    ROLL_WINDOWS = [7, 14, 28]

    for lag in LAG_DAYS:
        full[f'vol_lag_{lag}'] = full.groupby('HubID')['OrderVolume'].shift(lag)

    shifted_vol = full.groupby('HubID')['OrderVolume'].shift(1)
    for w in ROLL_WINDOWS:
        full[f'vol_roll{w}_mean'] = shifted_vol.groupby(full['HubID']).transform(
            lambda x: x.rolling(w, min_periods=1).mean())
        full[f'vol_roll{w}_std']  = shifted_vol.groupby(full['HubID']).transform(
            lambda x: x.rolling(w, min_periods=1).std())
        full[f'vol_roll{w}_max']  = shifted_vol.groupby(full['HubID']).transform(
            lambda x: x.rolling(w, min_periods=1).max())

    full['vol_expand_mean'] = shifted_vol.groupby(full['HubID']).transform(
        lambda x: x.expanding().mean())

    # AppSessions lags (not available in test, use historical lags)
    for lag in [7, 14, 28]:
        full[f'appsess_lag_{lag}'] = full.groupby('HubID')['AppSessions'].shift(lag)
    shifted_sess = full.groupby('HubID')['AppSessions'].shift(1)
    full['appsess_roll7_mean']  = shifted_sess.groupby(full['HubID']).transform(
        lambda x: x.rolling(7, min_periods=1).mean())
    full['appsess_roll14_mean'] = shifted_sess.groupby(full['HubID']).transform(
        lambda x: x.rolling(14, min_periods=1).mean())

    lag_cols = [c for c in full.columns if c.startswith(('vol_lag','vol_roll','vol_expand','appsess'))]
    cutoff   = train_df['Date'].max()
    return (
        full[full['Date'] <= cutoff][['HubID','Date'] + lag_cols],
        full[full['Date'] >  cutoff][['HubID','Date'] + lag_cols],
        lag_cols
    )


print('Computing lag & rolling features...')
train_lags, test_lags, lag_cols = add_lag_features(train_fe, test_fe)
train_fe = train_fe.merge(train_lags, on=['HubID','Date'], how='left')
test_fe  = test_fe.merge(test_lags,  on=['HubID','Date'], how='left')
print(f'Lag columns added: {len(lag_cols)}')

### 4.2 OOF Hub Target Encoding

In [ ]:
def oof_hub_target_encode(train_df, test_df, target='OrderVolume', n_splits=5):
    train_sorted = train_df.sort_values('Date').copy()
    oof_vals = np.zeros(len(train_sorted))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for _, (tr_idx, val_idx) in enumerate(tscv.split(train_sorted)):
        tr_fold = train_sorted.iloc[tr_idx]
        val_fold = train_sorted.iloc[val_idx]
        hub_means = tr_fold.groupby('HubID')[target].mean()
        global_mean = tr_fold[target].mean()
        oof_vals[val_idx] = val_fold['HubID'].map(hub_means).fillna(global_mean).values
    train_df = train_df.copy()
    train_df['hub_oof_mean_vol'] = oof_vals
    hub_means_full = train_df.groupby('HubID')[target].mean()
    test_df = test_df.copy()
    test_df['hub_oof_mean_vol'] = test_df['HubID'].map(hub_means_full).fillna(train_df[target].mean())
    return train_df, test_df


def hub_weekday_encode(train_df, test_df, target='OrderVolume'):
    open_train = train_df[train_df['IsOpen'] == 1]
    hw = open_train.groupby(['HubID', 'Weekday'])[target].mean().reset_index()
    hw.columns = ['HubID', 'Weekday', 'hub_weekday_mean_vol']
    train_df = train_df.merge(hw, on=['HubID','Weekday'], how='left')
    test_df  = test_df.merge(hw,  on=['HubID','Weekday'], how='left')
    hub_mean = open_train.groupby('HubID')[target].mean()
    train_df['hub_weekday_mean_vol'].fillna(train_df['HubID'].map(hub_mean), inplace=True)
    test_df['hub_weekday_mean_vol'].fillna(test_df['HubID'].map(hub_mean), inplace=True)
    return train_df, test_df


print('OOF hub encoding...')
train_fe, test_fe = oof_hub_target_encode(train_fe, test_fe)
print('Hub-weekday encoding...')
train_fe, test_fe = hub_weekday_encode(train_fe, test_fe)
print('Done')

### 4.3 Prepare Feature Matrix

In [ ]:
DROP_COLS = [
    'Date', 'OrderVolume', 'AppSessions',
    'CompetitorOpenSinceMonth', 'CompetitorOpenSinceYear',
    'LoyaltyProgramSinceWeek', 'LoyaltyProgramSinceYear',
    'LoyaltyProgramInterval', 'CompetitorDistance', 'Id',
]

# Only model open-hub rows
train_model = train_fe[train_fe['IsOpen'] == 1].copy()

FEATURE_COLS = [c for c in train_model.columns if c not in DROP_COLS and c != 'HubID']
print(f'Total features: {len(FEATURE_COLS)}')

# Drop rows where lags are NaN (unavoidable for early history)
key_lags = [c for c in lag_cols if c in train_model.columns][:3]  # first 3 lags
train_model_clean = train_model.dropna(subset=key_lags).sort_values('Date').reset_index(drop=True)
print(f'Train rows after lag NaN drop: {len(train_model_clean):,}')

X       = train_model_clean[FEATURE_COLS]
y       = np.log1p(train_model_clean['OrderVolume'])
y_raw   = train_model_clean['OrderVolume'].values
X_test  = test_fe[FEATURE_COLS]

print(f'X: {X.shape}, X_test: {X_test.shape}')

## 📐 5. RMSLE Metric

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

print('RMSLE function ready.')
print(f'Test metric on train mean predictor: {rmsle(y_raw, np.full_like(y_raw, y_raw.mean())):.5f}')

## 🌲 6. LightGBM

In [ ]:
LGB_PARAMS = {
    'objective': 'regression', 'metric': 'rmse',
    'learning_rate': 0.03, 'num_leaves': 127,
    'min_child_samples': 50, 'feature_fraction': 0.75,
    'bagging_fraction': 0.80, 'bagging_freq': 1,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_estimators': 3000, 'random_state': SEED,
    'verbosity': -1, 'n_jobs': -1,
}

N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

oof_lgb, pred_lgb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_lgb = []
lgb_fi = pd.Series(0.0, index=FEATURE_COLS)

print('Training LightGBM...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**LGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(500)])

    val_pred = np.maximum(np.expm1(model.predict(X_val)), 0)
    oof_lgb[val_idx] = val_pred
    score = rmsle(y_raw[val_idx], val_pred)
    cv_scores_lgb.append(score)
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration_}')

    pred_lgb += np.maximum(np.expm1(model.predict(X_test)), 0) / N_SPLITS
    lgb_fi   += pd.Series(model.feature_importances_, index=FEATURE_COLS) / N_SPLITS

print(f'\nLightGBM OOF RMSLE: {np.mean(cv_scores_lgb):.5f} +/- {np.std(cv_scores_lgb):.5f}')

In [ ]:
# Feature importance plot
top30 = lgb_fi.sort_values(ascending=False).head(30)
fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, 30)[::-1])
ax.barh(top30.index[::-1], top30.values[::-1], color=colors[::-1], alpha=0.9)
ax.set_title('LightGBM Top 30 Feature Importances', fontsize=13)
ax.set_xlabel('Importance')
ax.grid(True, axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

## 🚀 7. XGBoost

In [ ]:
XGB_PARAMS = {
    'objective': 'reg:squarederror', 'learning_rate': 0.03,
    'max_depth': 7, 'n_estimators': 3000,
    'subsample': 0.80, 'colsample_bytree': 0.75,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'min_child_weight': 50, 'tree_method': 'hist',
    'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
}

oof_xgb, pred_xgb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_xgb = []

print('Training XGBoost...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=100, eval_metric='rmse')
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=500)

    val_pred = np.maximum(np.expm1(model.predict(X_val)), 0)
    oof_xgb[val_idx] = val_pred
    score = rmsle(y_raw[val_idx], val_pred)
    cv_scores_xgb.append(score)
    print(f'  Fold {fold}: RMSLE={score:.5f}  best_iter={model.best_iteration}')

    pred_xgb += np.maximum(np.expm1(model.predict(X_test)), 0) / N_SPLITS

print(f'\nXGBoost OOF RMSLE: {np.mean(cv_scores_xgb):.5f} +/- {np.std(cv_scores_xgb):.5f}')

## 🐱 8. CatBoost

In [ ]:
CAT_FEATURES = [c for c in ['HubFormat','AssortmentTier','LoyaltyProgram',
                              'IsOpen','PromoActive','RegionalHoliday',
                              'SchoolClosureFlag','Weekday','IsWeekend','AnyHoliday']
                if c in FEATURE_COLS]

CB_PARAMS = {
    'iterations': 3000, 'learning_rate': 0.03, 'depth': 8,
    'l2_leaf_reg': 3, 'random_strength': 1, 'bagging_temperature': 1,
    'cat_features': CAT_FEATURES, 'eval_metric': 'RMSE',
    'early_stopping_rounds': 100, 'random_seed': SEED,
    'verbose': 500, 'thread_count': -1,
}

oof_cb, pred_cb = np.zeros(len(X)), np.zeros(len(X_test))
cv_scores_cb = []

print('Training CatBoost...')
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X), 1):
    X_tr, X_val = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    for col in CAT_FEATURES:
        X_tr[col], X_val[col] = X_tr[col].astype(str), X_val[col].astype(str)

    model = cb.CatBoostRegressor(**CB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    val_pred = np.maximum(np.expm1(model.predict(X_val)), 0)
    oof_cb[val_idx] = val_pred
    score = rmsle(y_raw[val_idx], val_pred)
    cv_scores_cb.append(score)
    print(f'  Fold {fold}: RMSLE={score:.5f}')

    X_test_cb = X_test.copy()
    for col in CAT_FEATURES: X_test_cb[col] = X_test_cb[col].astype(str)
    pred_cb += np.maximum(np.expm1(model.predict(X_test_cb)), 0) / N_SPLITS

print(f'\nCatBoost OOF RMSLE: {np.mean(cv_scores_cb):.5f} +/- {np.std(cv_scores_cb):.5f}')

## 🏆 9. Optimized Ensemble Blend

In [ ]:
oof_stack  = np.column_stack([oof_lgb, oof_xgb, oof_cb])
test_stack = np.column_stack([pred_lgb, pred_xgb, pred_cb])
model_names = ['LightGBM', 'XGBoost', 'CatBoost']

def blend_rmsle(weights):
    w = np.maximum(weights, 0)
    w = w / w.sum()
    return rmsle(y_raw, oof_stack @ w)

result = minimize(blend_rmsle, x0=[1/3, 1/3, 1/3],
                  method='Nelder-Mead', options={'maxiter': 2000, 'xatol': 1e-6})
weights = np.maximum(result.x, 0)
weights = weights / weights.sum()

oof_blend  = oof_stack  @ weights
test_blend = test_stack @ weights
blend_score = rmsle(y_raw, oof_blend)

print('Optimal weights:')
for name, w in zip(model_names, weights): print(f'  {name}: {w:.4f}')

print(f'\nOOF RMSLE Comparison:')
print(f'  LightGBM : {np.mean(cv_scores_lgb):.5f}')
print(f'  XGBoost  : {np.mean(cv_scores_xgb):.5f}')
print(f'  CatBoost : {np.mean(cv_scores_cb):.5f}')
print(f'  Ensemble : {blend_score:.5f}  <-- best')

In [ ]:
# OOF plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample_idx = np.random.choice(len(y_raw), min(5000, len(y_raw)), replace=False)

axes[0].scatter(y_raw[sample_idx], oof_blend[sample_idx], alpha=0.2, color=PALETTE[0], s=6)
mv = max(y_raw.max(), oof_blend.max())
axes[0].plot([0,mv],[0,mv],'r--', linewidth=1.5)
axes[0].set_title(f'Ensemble: Actual vs OOF Predicted (RMSLE={blend_score:.5f})')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].grid(True, alpha=0.3)

residuals = np.log1p(np.maximum(oof_blend, 0)) - np.log1p(y_raw)
axes[1].hist(residuals[sample_idx], bins=80, color=PALETTE[3], alpha=0.85, edgecolor='none')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('OOF Residuals (log space)')
axes[1].set_xlabel('log1p(pred) - log1p(actual)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 📤 10. Generate Submission

In [ ]:
test_blend = np.maximum(test_blend, 0)

# Build final predictions: IsOpen=0 -> 0, IsOpen=1 -> blended model
test_fe_reset = test_fe.reset_index(drop=True)
final_preds   = np.zeros(len(test_fe_reset))
open_mask     = (test_fe_reset['IsOpen'] == 1).values
final_preds[open_mask] = test_blend

# Match submission format
if 'Id' in test_fe_reset.columns:
    submission = pd.DataFrame({'Id': test_fe_reset['Id'].values, 'OrderVolume': final_preds})
else:
    submission = pd.DataFrame({'Id': sample['Id'].values, 'OrderVolume': final_preds})

submission['OrderVolume'] = submission['OrderVolume'].clip(lower=0).round(2)

print(f'Submission shape: {submission.shape}')
print(f'OrderVolume stats:')
print(submission['OrderVolume'].describe())
print(f'Zeros: {(submission["OrderVolume"]==0).sum()} ({(submission["OrderVolume"]==0).mean()*100:.1f}%)')

submission.head(10)

In [ ]:
submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(f'ID match with sample: {(submission["Id"].values == sample["Id"].values).all()}')

## 📊 11. Final Summary

In [ ]:
summary = pd.DataFrame({
    'Model':      model_names + ['Ensemble'],
    'OOF RMSLE': [np.mean(cv_scores_lgb), np.mean(cv_scores_xgb),
                  np.mean(cv_scores_cb), blend_score],
    'Weight':    [*weights, 1.0]
}).sort_values('OOF RMSLE').reset_index(drop=True)
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = [PALETTE[1] if m == 'Ensemble' else PALETTE[0] for m in summary['Model']]
bars = ax.bar(summary['Model'], summary['OOF RMSLE'], color=bar_colors, alpha=0.85, width=0.5)
ax.set_title('OOF RMSLE by Model', fontsize=13)
ax.set_ylabel('RMSLE (lower is better)')
ax.set_ylim(summary['OOF RMSLE'].min() * 0.98, summary['OOF RMSLE'].max() * 1.02)
ax.grid(True, axis='y', alpha=0.4)
for bar, val in zip(bars, summary['OOF RMSLE']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.0002, f'{val:.5f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## Next Steps to Lower RMSLE Further

| Improvement | Expected Gain |
|---|---|
| Optuna HPO on LightGBM | -0.005 to -0.010 |
| Longer lags (90, 365 days for YoY) | -0.003 |
| Hub cluster features (K-Means) | -0.002 |
| Per-hub Prophet blending | -0.002 |
| 4th model (TabNet / MLP) | -0.002 |